# Caption v2 — higher-quality octopus captions + ethogram (Qwen3-VL-30B)

Fixes the low-quality captions from `caption_octopus_clips.ipynb` by improving the
**input to the VLM** (the real bottleneck on dim IR footage), not just the prompt:

1. **Frame enhancement** — CLAHE brightness/contrast on each frame so the octopus is
   visible instead of a dark blob.
2. **Higher resolution** — frames extracted large and sent at up to ~768px (was 512).
3. **Best-frame selection** — score every candidate frame with the CLIP+MLP probe
   (`clip_mlp_hardneg_v2.pt`) and send Qwen the **top-K frames where the octopus is
   most visible**, in time order (instead of 10 uniform, often-empty frames).
4. **Skip non-octopus clips** — if no frame clears the presence threshold, label
   `octopus not present` and skip the VLM (no hallucinated captions, saves compute).
5. **Allow "uncertain"** — the model may abstain on the ethogram instead of guessing,
   so the labels you keep are trustworthy.

Writes `caption` + `ethogram_label` (+ `caption_pipeline="v2-enhanced"`, `max_p_visible`).
**Resumable & non-destructive**: skips clips already done by v2 AND clips you've
`approved` in review (won't clobber human-approved captions).

> Runtime → A100 GPU. Uses Qwen3-VL-30B (vLLM) + CLIP ViT-B/32 together.

> **src/ pipeline step**: this fills `caption` + `ethogram_label` (7-class `ethogram_list_v2.json`) into `src/octopus_clips_verified.json` — the separate captioning step the extractor leaves for later. Upload that JSON (+ clips zip + `ethogram_list_v2.json` + `clip_mlp_hardneg_v2.pt`) to Drive, run here, copy the JSON back into `src/`.

## 1. Install

In [ ]:
!pip uninstall -y -q torch torchvision torchaudio torchao 2>/dev/null
!pip install -q -U vllm qwen-vl-utils openai-clip opencv-python-headless
!apt-get -qq install -y ffmpeg >/dev/null
print("Installed. NOW: Runtime -> Restart session, then run from the CONFIG cell.")

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Config

In [ ]:
from pathlib import Path

MODEL         = "QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ"

# frame handling
DENSE_FPS     = 1.0      # candidate frames sampled per second (~20 for a 20s clip)
N_KEEP        = 8        # how many best frames to send to Qwen
EXTRACT_W     = 1280     # long-side px for extracted frames (source is 3840; keep detail)
QWEN_MAXPIX   = 768*768  # per-frame cap sent to Qwen (was 512*512)
VIS_THRESH    = 0.6      # per-frame p_visible to count a frame as "octopus visible"
PRESENT_MIN   = 0.5      # if NO frame reaches this p_visible -> treat clip as no-octopus (tunable; raise=stricter)
ENHANCE       = True     # CLAHE brightness/contrast before sending to Qwen

MAX_TOKENS    = 220
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL  = 0.85     # leave room for CLIP alongside vLLM
IMAGE_LIMIT   = 16

INDEX_JSON    = Path("octopus_clips_verified.json")
CLIPS_ROOT    = Path("octopus_clips_verified")
ETHOGRAM_PATH = Path("ethogram_list_v2.json")   # 7-class compact sheet (matches training)
CLIP_CKPT     = Path("clip_mlp_hardneg_v2.pt")     # CLIP+MLP probe for frame scoring
PIPELINE_TAG  = "v2-enhanced"

## 3. Get data into Colab
Put on Drive (`MyDrive/GSOC-Catrobat/`): `octopus_clips_verified.zip`,
`octopus_clips_verified.json` (from **src/**), `ethogram_list_v2.json`, and `clip_mlp_hardneg_v2.pt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil
DRIVE_ROOT = Path("/content/drive/MyDrive/GSOC-Catrobat")
with zipfile.ZipFile(DRIVE_ROOT / "octopus_clips_verified.zip") as z:
    z.extractall(".")
shutil.copy(DRIVE_ROOT / "octopus_clips_verified.json", INDEX_JSON)
shutil.copy(DRIVE_ROOT / "ethogram_list_v2.json", ETHOGRAM_PATH)
shutil.copy(DRIVE_ROOT / "clip_mlp_hardneg_v2.pt", CLIP_CKPT)
print(len(list(CLIPS_ROOT.rglob("*.mp4"))), "clips ready")

## 4. Load models — Qwen3-VL (vLLM) + CLIP+MLP probe

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

print(f"Loading {MODEL} (vLLM) ...")
llm = LLM(model=MODEL, max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
          limit_mm_per_prompt={"image": IMAGE_LIMIT}, dtype="auto", trust_remote_code=True)
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
print("Qwen ready.")

In [ ]:
# CLIP ViT-B/32 + MLP probe, for per-frame octopus scoring (letterbox preprocessing)
try:
    import pkg_resources, packaging, packaging.version, packaging.specifiers, packaging.requirements
    pkg_resources.packaging = packaging
except Exception: pass
import clip as clip_lib
import torch, torch.nn as nn

cdev = "cuda"
_ck = torch.load(CLIP_CKPT, map_location=cdev)
clip_model, _clip_pre = clip_lib.load(_ck["clip_model"], device=cdev); clip_model.eval()
def _build_clf(ck):
    feat=ck["feat_dim"]; hid=[int(x) for x in ck["arch"].replace("mlp_","").split("_")]; dims=[feat]+hid+[2]
    L=[]
    for i in range(len(dims)-1):
        L.append(nn.Linear(dims[i],dims[i+1]))
        if i<len(dims)-2: L+=[nn.ReLU(), nn.Dropout(0.3)]
    return nn.Sequential(*L)
clf = _build_clf(_ck).to(cdev); clf.load_state_dict(_ck["state_dict"]); clf.eval()
VIS_IDX = _ck.get("label_map",{}).get("visible",1)
print(f"CLIP probe ready ({_ck['clip_model']}+{_ck['arch']}, acc {_ck.get('test_acc',0):.1%})")

## 5. Frame extraction, enhancement, scoring, selection

In [ ]:
import subprocess, tempfile, cv2, numpy as np
from PIL import Image

def letterbox(img, size=224, fill=(128,128,128)):
    w,h=img.size; s=size/max(w,h); nw,nh=max(1,round(w*s)),max(1,round(h*s))
    img=img.resize((nw,nh),Image.BICUBIC); cv=Image.new("RGB",(size,size),fill)
    cv.paste(img,((size-nw)//2,(size-nh)//2)); return cv

def enhance(img):
    """CLAHE brightness/contrast on the L channel so the dark IR octopus is visible."""
    a=np.array(img.convert("RGB")); lab=cv2.cvtColor(a,cv2.COLOR_RGB2LAB)
    l,A,B=cv2.split(lab); l=cv2.createCLAHE(clipLimit=2.5,tileGridSize=(8,8)).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((l,A,B)),cv2.COLOR_LAB2RGB))

def extract_dense(clip_path, tmpdir):
    pat=str(Path(tmpdir)/"f_%03d.jpg")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(clip_path),
        "-vf",f"fps={DENSE_FPS},scale='min({EXTRACT_W},iw)':-2","-q:v","2",pat],check=True)
    return sorted(str(p) for p in Path(tmpdir).glob("f_*.jpg"))

def score_frames(paths):
    """Per-frame p_visible from the CLIP+MLP probe (letterbox, no enhancement — matches training)."""
    ps=[]
    for i in range(0,len(paths),64):
        batch=[]
        for p in paths[i:i+64]:
            try: batch.append(_clip_pre(letterbox(Image.open(p).convert("RGB"))))
            except Exception: batch.append(torch.zeros(3,224,224))
        with torch.no_grad():
            f=clip_model.encode_image(torch.stack(batch).to(cdev)).float(); f=f/f.norm(dim=-1,keepdim=True)
            p=torch.softmax(clf(f),dim=1)[:,VIS_IDX]
        ps.extend(p.cpu().tolist())
    return ps

def select_best(paths, scores):
    """Top-N_KEEP frames by p_visible, returned in time order."""
    order=sorted(range(len(paths)), key=lambda i:scores[i], reverse=True)[:N_KEEP]
    return [paths[i] for i in sorted(order)]

## 6. Prompt + parsing (allows 'uncertain')

In [ ]:
import json
ethogram=json.load(open(ETHOGRAM_PATH)); behaviors=ethogram["behaviors"]
labels=[b["label"] for b in behaviors]; valid_set=set(labels)
label_block="\n".join(f"- {b['label']}: {b['description']}" for b in behaviors)

def build_prompt():
    return (
        "These are the CLEAREST, brightness-enhanced frames from one 20s aquarium clip, in time order. "
        "Subject: Nity, an octopus (Octopus vulgaris) in a dim IR-lit tank. "
        "Describe ONLY what you can actually see — do not invent details.\n\n"
        "1) If you genuinely cannot see an octopus in any frame, respond EXACTLY:\n"
        "   CAPTION: octopus not present\n   ETHOGRAM: octopus not present\n"
        "2) Otherwise write ONE caption of what the octopus does across the clip (movement, posture, "
        "arm position, color/texture, and any object or human it interacts with), then choose the single "
        "best behavior label below. If the behavior is genuinely unclear, use 'uncertain' — do NOT guess.\n"
        f"{label_block}\n\n"
        "Respond in EXACTLY this format, nothing else:\n"
        "CAPTION: <one sentence>\n"
        "ETHOGRAM: <one label verbatim, or 'uncertain', or 'octopus not present'>"
    )

_KW=[(["crawl","walking on arms","moving across"],"Crawling"),(["swim","jet","propel","water column"],"Swimming / jetting"),
(["arm walk","bipedal"],"Arm walking"),(["hunt","stalk","pursuit"],"Hunting"),(["captur","pounce","grab","seiz"],"Capturing prey"),
(["eat","feeding","tearing food","food"],"Manipulating food"),(["entering den","into den","retreating into"],"Entering den"),
(["exiting den","emerging","leaving den"],"Exiting den"),(["rearrang","piling","moving shells","moving rocks"],"Rearranging den"),
(["extend","probing","reaching out","arm out"],"Arm extension / probing"),(["manipulat","picking up","holding object","playing with"],"Object manipulation"),
(["above water","out of water","water surface"],"Reaching out of water"),(["human","person","hand","researcher"],"Responding to human"),
(["joystick","toy","enrichment","device","screen"],"Enrichment interaction"),(["color","colour","blanch","darken","texture","camouflage"],"Color / texture change"),
(["ink","cloud"],"Ink release"),(["hid","flatten","conceal","press"],"Hiding / flattening"),
(["sitting in den","inside den","resting in den"],"Stationary in den"),(["open area","tank floor","in the open"],"Stationary in open")]
def match_ethogram(t):
    t=t.lower()
    for kws,lab in _KW:
        if any(k in t for k in kws): return lab
    return "uncertain"

def parse_response(text):
    caption,etho="",None
    for line in text.splitlines():
        s=line.strip()
        if s.upper().startswith("CAPTION:"): caption=s[8:].strip().strip("'\"")
        elif s.upper().startswith("ETHOGRAM:"):
            raw=s[9:].strip().strip("'\""); rl=raw.lower()
            if rl in ("uncertain","not sure","unknown"): etho="uncertain"
            elif "not present" in rl: etho="octopus not present"
            else:
                for lab in valid_set:
                    if lab.lower()==rl or lab.lower() in rl or rl in lab.lower(): etho=lab; break
                else: etho=raw or None
    if not caption: caption=text.strip().strip("'\"")
    if "not present" in caption.lower() or "not visible" in caption.lower(): return "octopus not present","octopus not present"
    if not etho: etho=match_ethogram(caption)
    elif etho not in valid_set and etho not in ("uncertain","octopus not present"): etho=match_ethogram(f"{caption} {etho}")
    return caption,etho
print(f"{len(labels)} labels loaded.")

## 7. Caption a single clip (enhanced high-res best frames)

In [ ]:
from qwen_vl_utils import process_vision_info

def caption_best_frames(frame_paths, tmpdir, prompt):
    imgs=[]
    for j,p in enumerate(frame_paths):
        im=Image.open(p).convert("RGB")
        if ENHANCE: im=enhance(im)
        q=str(Path(tmpdir)/f"q_{j:02d}.jpg"); im.save(q,quality=92); imgs.append(q)
    content=[{"type":"image","image":p,"max_pixels":QWEN_MAXPIX} for p in imgs]
    content.append({"type":"text","text":prompt})
    messages=[{"role":"user","content":content}]
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs,_=process_vision_info(messages)
    out=llm.generate({"prompt":text,"multi_modal_data":{"image":image_inputs}},
                     sampling_params=sampling_params,use_tqdm=False)
    return out[0].outputs[0].text.strip()

def resolve_clip(entry):
    rel=entry["clip_path"].split("octopus_clips_verified/",1)[-1]
    return CLIPS_ROOT/rel

## 8. Run (resumable, non-destructive)
Skips clips already done by v2 and clips you've `approved` in review.

In [ ]:
from datetime import datetime
from collections import Counter

prompt=build_prompt()
index=json.load(open(INDEX_JSON)); clips=index["clips"]
todo=[c for c in clips if c.get("caption_pipeline")!=PIPELINE_TAG and c.get("review")!="approved"]
print(f"{len(clips)} clips, {len(todo)} to (re)caption with v2\n"+"-"*60)

done=skipped_absent=0
for i,e in enumerate(todo):
    cp=resolve_clip(e); print(f"[{i+1}/{len(todo)}] {e['clip_path']}",flush=True)
    if not cp.exists(): print("  ! missing file"); continue
    with tempfile.TemporaryDirectory() as tmp:
        try:
            frames=extract_dense(cp,tmp)
            if not frames: print("  no frames"); continue
            scores=score_frames(frames)
            maxp=max(scores)
            e["max_p_visible"]=round(maxp,4)
            if maxp < PRESENT_MIN:                       # no real octopus -> skip VLM
                e["caption"]="octopus not present"; e["ethogram_label"]="octopus not present"
                skipped_absent+=1; verdict="octopus not present (skipped VLM)"
            else:
                best=select_best(frames,scores)
                raw=caption_best_frames(best,tmp,prompt)
                cap,etho=parse_response(raw)
                e["caption"]=cap; e["ethogram_label"]=etho; verdict=f"{etho}"
        except Exception as ex:
            print(f"  failed: {ex}"); continue
    e["caption_model"]=MODEL.split("/")[-1]; e["caption_pipeline"]=PIPELINE_TAG
    e["captioned_at"]=datetime.now().isoformat(timespec="seconds")
    print(f"  max_p={e['max_p_visible']}  -> {verdict}")
    if e.get('caption'): print(f"  {e['caption'][:110]}",flush=True)
    with open(INDEX_JSON,"w") as f: json.dump(index,f,indent=2)
    done+=1

print("-"*60+f"\nDone. recaptioned {done} ({skipped_absent} auto no-octopus).")
print("\nEthogram distribution (v2 clips):")
for lab,n in Counter(c.get("ethogram_label") for c in clips if c.get("caption_pipeline")==PIPELINE_TAG).most_common():
    print(f"  {n:4d}  {lab}")

## 9. Save the JSON back

In [ ]:
import shutil
if Path("/content/drive/MyDrive").exists():
    shutil.copy(INDEX_JSON, DRIVE_ROOT/INDEX_JSON.name); print("Copied back to Drive.")
else:
    from google.colab import files; files.download(str(INDEX_JSON))